> **Catatan (diperbarui):** notebook ini adalah catatan eksplorasi dan analisis awal. Pipeline final yang dipakai aplikasi ada di folder `src/` dan dijalankan lewat `retrain.py`. Angka final (MAPE, rentang 80%, dan pilihan model vs harga bulan lalu per komoditas) ada di folder `results/`. Beberapa sel di bawah mungkin menggunakan kriteria seleksi lama (MAE) dan fitur lama (`geo_id`, `month`).

# 05 - Evaluasi Model
Membaca hasil yang dihasilkan `retrain.py` (results/ dan models/).

In [ ]:
import sys, os
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
import pandas as pd, numpy as np
from IPython.display import Image, display
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)

In [ ]:
import json, joblib
meta = json.load(open('models/model_metadata.json'))
print(json.dumps({k: meta[k] for k in ['model_name','train_period','validation_period','test_period','metrics_test','naive_baseline_test']}, indent=1))

In [ ]:
r = pd.read_csv('results/evaluation_metrics.csv')
r[r.split=='test'].sort_values('MAE')[['model','MAE','RMSE','R2','MAPE_%']].round(3)

In [ ]:
c = pd.read_csv('results/evaluation_by_commodity.csv')
c[c.model.isin([meta['model_name'],'Naive Forecast'])].round(2)

In [ ]:
for f in sorted(Path('results/figures').glob('eval_*.png')):
    display(Image(str(f)))

In [ ]:
# Verifikasi model tersimpan dapat dimuat dan konsisten dengan hasil evaluasi
from src.feature_engineering import build_features
from src.data_loader import load_long
from src import training as T
b = joblib.load('models/best_model.pkl')
_, _, test = T.make_splits(build_features(load_long()))
p = T.predict_strategy(b['models'], test, b['strategy'], b['target_mode'])
saved = pd.read_csv('results/prediction_results.csv')[meta['model_name']].values
print('Konsisten:', np.allclose(p, saved, atol=0.01), '| NaN:', np.isnan(p).sum())